## 重回帰分析の演習

この演習では，架空の臨床データを用いて重回帰分析を実行し，複数の説明変数が目的変数に与える影響を評価する方法を学びます．

本演習の目的は以下の3点です．

1.   重回帰分析の基本的な実行方法を理解する
2.   回帰係数・p値・決定係数（R²）の意味を理解する

本Notebookは，上から順番に実行することで解析を再現できるよう構成されています．


##　架空の臨床データの作成
実際の研究データの代わりに，架空の患者データを作成します．

今回は以下の変数を生成します．

*   年齢（歳）
*   BMI（kg/m²）
*   喫煙歴（あり・なし）
*   収縮期血圧（mmHg）

収縮期血圧は，年齢やBMIの影響を受けるように設定されています．

実際の研究では観察データを用いますが，本演習では変数間の関係を理解しやすくするために人工的なデータを作成しています．


In [4]:
# ============================================
# 架空の臨床データを作成するためのプログラム
# ============================================

import numpy as np
import pandas as pd

# 乱数を固定してデータの再現性を高める
np.random.seed(42)

# サンプルサイズ
n = 300

# 説明変数の設定
age = np.random.normal(60, 12, n)          # 年齢
bmi = np.random.normal(24, 3.5, n)         # BMI kg/m²
smoking = np.random.binomial(1, 0.3, n)    # 喫煙歴　0=no, 1=yes
diabetes = np.random.binomial(1, 0.2, n)   # 糖尿病　0=no, 1=yes

# ランダムエラー
error = np.random.normal(0, 8, n)

# 目的変数 (収縮期血圧)
sbp = (
    70
    + 0.5 * age
    + 1.2 * bmi
    + 5.0 * smoking
    + 8.0 * diabetes
    + error
)

# データフレームとしてデータを作成する
df = pd.DataFrame({
    "Age": age,
    "BMI": bmi,
    "Smoking": smoking,
    "Diabetes": diabetes,
    "SBP": sbp
})

# CSV形式で保存する
df.to_csv("synthetic_clinical_data.csv", index=False)



## データの確認

作成したデータの先頭数行を表示します．

各列がどの変数を表しているか確認してください．

研究データを解析する際には，まずデータの内容や欠損値の有無を確認することが重要です．

In [ ]:
# データの確認
print(df.head())
print("\nData shape:", df.shape)

## 重回帰分析を実行する
ここでは，

- 年齢
- BMI
- 喫煙歴

を説明変数，

- 収縮期血圧

を目的変数として重回帰分析を行います．

重回帰分析では，他の変数の影響を調整したうえで，各変数が独立して目的変数と関連しているかを評価できます．

In [8]:
# ==========================================
# 重回帰分析のプログラム
# ==========================================

import statsmodels.api as sm

# データの読込み
df = pd.read_csv("synthetic_clinical_data.csv")

# 目的変数
y = df["SBP"]

# 説明変数
X = df[["Age", "BMI", "Smoking", "Diabetes"]]

# 切片を追加
X = sm.add_constant(X)

# 重回帰モデルを推定
model = sm.OLS(y, X).fit()


## 結果の読み方

出力結果には多くの数値が表示されますが，
まず以下の項目に注目してください．

### 回帰係数（coef）

説明変数が1単位増加したときに，
目的変数が平均でどれだけ変化するかを表します．

### p値

観察された関連が偶然で説明できる可能性を示します．

一般には

p < 0.05

で統計学的に有意と判断されます．

### R²（決定係数）

モデルが目的変数の変動をどの程度説明できるかを示します．

0に近いほど説明力が低く，
1に近いほど説明力が高いことを意味します．

### adjusted R²

説明変数の数を考慮した決定係数です．

モデル同士を比較する際はこちらを重視します．

In [ ]:
# 結果を表示
print(model.summary())

# 回帰係数だけを見やすく表示する
results = pd.DataFrame({
    "Coefficient": model.params,
    "95% CI Lower": model.conf_int()[0],
    "95% CI Upper": model.conf_int()[1],
    "P-value": model.pvalues
})
print(results.round(3))

## 単回帰分析との比較
まず，単回帰分析では例えば以下のようなモデルを実行する．

*   目的変数：収縮期血圧（SBP）
*   説明変数：年齢（Age）

このときの出力には，主に次の項目が表示される．

*   回帰係数（coef）
*   p値
*   決定係数（R²）

ここでの回帰係数は，「年齢が1歳増えると血圧が平均どれだけ変化するか」を表す．

ただしこの結果は，「年齢だけの影響」ではなく，他の要因（BMIや喫煙など）を考慮していない点に注意が必要である．



In [ ]:
# ==========================================
# 単回帰分析（年齢のみを選択）
# ==========================================

import statsmodels.api as sm

# データの読込み
df = pd.read_csv("synthetic_clinical_data.csv")

# 説明変数を選択
X_simple = sm.add_constant(df["Age"])

# 単回帰分析のモデルを推定
model_simple = sm.OLS(df["SBP"], X_simple).fit()

# 結果の表示
print(model_simple.summary())

## 回帰係数の意味の違い（重要）

■ 単回帰の場合（Ageのみ）
*   年齢が1歳増えるときの血圧の変化
*   ただしBMIや喫煙の影響は含まれている可能性がある

■ 重回帰の場合（Age + BMI）
*   BMIが同じ条件で，年齢が1歳増えたときの血圧の変化
*   他の変数の影響を取り除いた「独立した効果」

## 単回帰と重回帰の結果が変わる理由

単回帰と重回帰で回帰係数やp値が変わるのは，「他の変数を考慮しているかどうか」が違うためである．

特に以下のような場合に差が大きくなる．

*   年齢とBMIが関連している
*   喫煙と年齢が関連している
*   複数の要因が同じ目的変数に影響している

このような状況では，単回帰の結果は「見かけの関係」になりやすく，
重回帰では「他の要因を調整した関係」が得られる．


